In [3]:
# =========================
# IMPORTS
# =========================
import tensorflow as tf
import numpy as np
import os
import zipfile
import urllib.request
import shutil

print("="*70)
print("TINY IMAGENET DATASET SETUP")
print("="*70)

# =========================
# PATHS
# =========================
data_dir = './tiny-imagenet-200'
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val')

# =========================
# DOWNLOAD DATASET
# =========================
if not os.path.exists(data_dir):
    print("Downloading Tiny ImageNet...")
    url = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'
    zip_path = 'tiny-imagenet-200.zip'
    urllib.request.urlretrieve(url, zip_path)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('./')

    os.remove(zip_path)
    print("Download & extraction complete")
else:
    print("Dataset already exists")

# =========================
# FIX VALIDATION STRUCTURE
# =========================
val_annotations_file = os.path.join(val_dir, 'val_annotations.txt')
val_images_dir = os.path.join(val_dir, 'images')

if os.path.exists(val_annotations_file):
    print("Fixing validation folder structure...")

    val_map = {}
    with open(val_annotations_file, 'r') as f:
        for line in f:
            img, cls = line.split('\t')[:2]
            val_map[img] = cls

    for img, cls in val_map.items():
        cls_dir = os.path.join(val_dir, cls)
        os.makedirs(cls_dir, exist_ok=True)

        src = os.path.join(val_images_dir, img)
        dst = os.path.join(cls_dir, img)
        if os.path.exists(src):
            shutil.move(src, dst)

    # ❗ IMPORTANT FIX (ROOT CAUSE)
    shutil.rmtree(val_images_dir, ignore_errors=True)
    print("Validation structure fixed")

# =========================
# REMOVE TRAIN IMAGES FOLDER (ROOT CAUSE FIX)
# =========================
train_images_path = os.path.join(train_dir, 'images')
shutil.rmtree(train_images_path, ignore_errors=True)

# =========================
# DATA GENERATORS
# =========================
IMG_SIZE = (128, 128)     # COLAB SAFE
BATCH_SIZE = 16          # COLAB SAFE

train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)

val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("Classes:", train_generator.num_classes)

# =========================
# MODEL (INCEPTION STYLE)
# =========================
inputs = tf.keras.layers.Input(shape=(128, 128, 3))

x = tf.keras.layers.Conv2D(64, 7, strides=2, padding='same', activation='relu')(inputs)
x = tf.keras.layers.MaxPooling2D(3, strides=2, padding='same')(x)

x = tf.keras.layers.Conv2D(64, 1, activation='relu')(x)
x = tf.keras.layers.Conv2D(192, 3, padding='same', activation='relu')(x)
x = tf.keras.layers.MaxPooling2D(3, strides=2, padding='same')(x)

# Inception block
def inception(x, f1, f3r, f3, f5r, f5, fp):
    b1 = tf.keras.layers.Conv2D(f1, 1, activation='relu')(x)

    b2 = tf.keras.layers.Conv2D(f3r, 1, activation='relu')(x)
    b2 = tf.keras.layers.Conv2D(f3, 3, padding='same', activation='relu')(b2)

    b3 = tf.keras.layers.Conv2D(f5r, 1, activation='relu')(x)
    b3 = tf.keras.layers.Conv2D(f5, 5, padding='same', activation='relu')(b3)

    b4 = tf.keras.layers.MaxPooling2D(3, strides=1, padding='same')(x)
    b4 = tf.keras.layers.Conv2D(fp, 1, activation='relu')(b4)

    return tf.keras.layers.concatenate([b1, b2, b3, b4])

x = inception(x, 64, 96, 128, 16, 32, 32)
x = inception(x, 128, 128, 192, 32, 96, 64)

x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.4)(x)

outputs = tf.keras.layers.Dense(200, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

# =========================
# COMPILE
# =========================
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# =========================
# TRAIN
# =========================
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=1,
    verbose=1
)

# =========================
# SAVE
# =========================
model.save("inception_tinyimagenet_fixed.h5")
print("Training complete & model saved")


TINY IMAGENET DATASET SETUP
Dataset already exists
Fixing validation folder structure...
Validation structure fixed
Found 100000 images belonging to 200 classes.
Found 10000 images belonging to 200 classes.
Classes: 200


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │      9,472 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 32, 32,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │      4,160 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │    110,784 │ conv2d_1[0][0]    │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 16, 16,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │     18,528 │ max_pooling2d_1[… │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 16, 16,    │      3,088 │ max_pooling2d_1[… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ max_pooling2d_1[… │
│ (MaxPooling2D)      │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 16, 16,    │     12,352 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 16, 16,    │    110,720 │ conv2d_4[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 16, 16,    │     12,832 │ conv2d_6[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 16, 16,    │      6,176 │ max_pooling2d_2[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16, 16,    │          0 │ conv2d_3[0][0],   │
│ (Concatenate)       │ 256)              │            │ conv2d_5[0][0],   │
│                     │                   │            │ conv2d_7[0][0],   │
│                     │                   │            │ conv2d_8[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 16, 16,    │     32,896 │ concatenate[0][0] │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 16, 16,    │      8,224 │ concatenate[0][0] │
│                     │ 32)               │            │                 

 Total params: 773,048 (2.95 MB)

 Trainable params: 773,048 (2.95 MB)

 Non-trainable params: 0 (0.00 B)

6250/6250 ━━━━━━━━━━━━━━━━━━━━ 475s 75ms/step - accuracy: 0.0088 - loss: 5.2316 - val_accuracy: 0.0304 - val_loss: 4.8758


Training complete & model saved


In [4]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.preprocessing import image

# Load trained model
model = tf.keras.models.load_model("inception_tinyimagenet_fixed.h5")

print("Model loaded successfully ✅")


Model loaded successfully ✅


In [5]:
# Re-create class index mapping from training directory
train_dir = "./tiny-imagenet-200/train"

class_names = sorted([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])

print("Total classes:", len(class_names))  # must be 200


Total classes: 200


In [7]:
# Path of outside image (change this)
img_path = "/content/cat.jpg"   # example

# Load image
img = image.load_img(img_path, target_size=(128, 128))

# Convert to array
img_array = image.img_to_array(img)

# Normalize
img_array = img_array / 255.0

# Add batch dimension (VERY IMPORTANT)
img_array = np.expand_dims(img_array, axis=0)

print("Image preprocessed successfully")


Image preprocessed successfully


In [8]:
# Predict
predictions = model.predict(img_array)

# Get predicted class index
predicted_class_index = np.argmax(predictions)

# Get confidence
confidence = np.max(predictions)

# Get class name
predicted_class_name = class_names[predicted_class_index]

print("Predicted Class ID :", predicted_class_index)
print("Predicted Class    :", predicted_class_name)
print(f"Confidence         : {confidence*100:.2f}%")


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted Class ID : 176
Predicted Class    : n06596364
Confidence         : 2.73%


In [9]:
# Load WordNet to human-readable label mapping
words_file = "./tiny-imagenet-200/words.txt"

id_to_label = {}

with open(words_file, "r") as f:
    for line in f:
        wnid, label = line.strip().split("\t")
        id_to_label[wnid] = label


In [10]:
# Predict
predictions = model.predict(img_array)

# Class index
predicted_class_index = np.argmax(predictions)

# Confidence
confidence = np.max(predictions)

# WordNet ID
predicted_wnid = class_names[predicted_class_index]

# Human-readable label
predicted_label = id_to_label.get(predicted_wnid, "Unknown")

print("Predicted WordNet ID :", predicted_wnid)
print("Predicted Class     :", predicted_label)
print(f"Confidence          : {confidence*100:.2f}%")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
Predicted WordNet ID : n06596364
Predicted Class     : comic book
Confidence          : 2.73%
